<a href="https://colab.research.google.com/github/Khushibung05/RAG/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 28.9 MB/s eta 0:00:00


In [3]:
from reportlab.pdfgen import canvas

In [4]:
from reportlab.pdfgen import canvas

documents = {

    "Employee Handbook.pdf": """
Employee Handbook

Employees are expected to maintain professional conduct.
Office timings are 9 AM to 6 PM.
Employees must follow company ethics and compliance policies.
""",

    "Leave Policy.pdf": """
Leave Policy

Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
""",

    "Travel Policy.pdf": """
Travel Policy

Travel expenses are reimbursed within 30 days.
Original receipts must be submitted for reimbursement.
Business travel must be approved by the reporting manager.
""",

    "Work From Home Policy.pdf": """
Work From Home Policy

Employees may work from home twice per week.
Manager approval is required for additional remote work.
Employees must remain available during working hours.
""",

    "Medical Insurance Policy.pdf": """
Medical Insurance Policy

All employees are covered under company medical insurance.
Dependent coverage is available for spouse and children.
Emergency hospitalization expenses are covered.
"""
}

for filename, content in documents.items():

    c = canvas.Canvas(filename)

    y = 800

    for line in content.split("\n"):
        c.drawString(50, y, line)
        y -= 20

    c.save()

print("All PDFs created successfully!")

All PDFs created successfully!


In [6]:
!pip install PyPDF2
from PyPDF2 import PdfReader
import os

pdf_files = [
    "Employee Handbook.pdf",
    "Leave Policy.pdf",
    "Travel Policy.pdf",
    "Work From Home Policy.pdf",
    "Medical Insurance Policy.pdf"
]

all_documents = []

for file in pdf_files:

    reader = PdfReader(file)

    text = ""

    for page in reader.pages:
        extracted = page.extract_text()

        if extracted:
            text += extracted + "\n"

    all_documents.append(text)

    print("="*60)
    print("File Name:", file)
    print("Number of Pages:", len(reader.pages))
    print("Total Characters:", len(text))
    print("Total Words:", len(text.split()))
    print("="*60)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.9 MB/s eta 0:00:00
File Name: Employee Handbook.pdf
Number of Pages: 1
Total Characters: 171
Total Words: 25
File Name: Leave Policy.pdf
Number of Pages: 1
Total Characters: 150
Total Words: 21
File Name: Travel Policy.pdf
Number of Pages: 1
Total Characters: 176
Total Words: 25
File Name: Work From Home Policy.pdf
Number of Pages: 1
Total Characters: 179
Total Words: 27
File Name: Medical Insurance Policy.pdf
Number of Pages: 1
Total Characters: 190
Total Words: 24


In [8]:
!pip install langchain-text-splitters
#fixed-size chunking
from langchain_text_splitters import CharacterTextSplitter

all_text = "\n".join(all_documents)

splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_text(all_text)

print("Total Chunks:", len(chunks))

for i, chunk in enumerate(chunks, start=1):

    print("\n")
    print("="*60)
    print(f"Chunk {i}")
    print("="*60)

    print(chunk)

    print("\nChunk Length:", len(chunk))

Total Chunks: 2


Chunk 1
Employee Handbook
Employees are expected to maintain professional conduct.
Office timings are 9 AM to 6 PM.
Employees must follow company ethics and compliance policies.
Leave Policy
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
Travel Policy
Travel expenses are reimbursed within 30 days.
Original receipts must be submitted for reimbursement.
Business travel must be approved by the reporting manager.

Chunk Length: 493


Chunk 2
Business travel must be approved by the reporting manager.
Work From Home Policy
Employees may work from home twice per week.
Manager approval is required for additional remote work.
Employees must remain available during working hours.
Medical Insurance Policy
All employees are covered under company medical insurance.
Dependent coverage is available for spouse and children.
Emergency hospitalization expenses are covered.

Chunk Length: 425


In [9]:
#recursive chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

all_text = "\n".join(all_documents)

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

recursive_chunks = recursive_splitter.split_text(all_text)

print("Total Chunks:", len(recursive_chunks))

for i, chunk in enumerate(recursive_chunks, start=1):

    print("\n")
    print("="*60)
    print(f"Recursive Chunk {i}")
    print("="*60)

    print(chunk)

    print("\nChunk Length:", len(chunk))

Total Chunks: 2


Recursive Chunk 1
Employee Handbook
Employees are expected to maintain professional conduct.
Office timings are 9 AM to 6 PM.
Employees must follow company ethics and compliance policies.


Leave Policy
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.


Travel Policy
Travel expenses are reimbursed within 30 days.
Original receipts must be submitted for reimbursement.
Business travel must be approved by the reporting manager.

Chunk Length: 497


Recursive Chunk 2
Work From Home Policy
Employees may work from home twice per week.
Manager approval is required for additional remote work.
Employees must remain available during working hours.


Medical Insurance Policy
All employees are covered under company medical insurance.
Dependent coverage is available for spouse and children.
Emergency hospitalization expenses are covered.

Chunk Length: 368


print("""
WHY CHUNKING IS NECESSARY IN RAG?

1. LLMs have context window limits and cannot process large documents efficiently.

2. Chunking breaks large documents into smaller meaningful units.

3. Retrieval becomes faster because only relevant chunks are searched.

4. Embeddings generated on smaller chunks capture information more accurately.

5. Chunk overlap helps preserve context between neighboring chunks.

FIXED SIZE CHUNKING:
- Easy to implement.
- May split sentences and lose meaning.

RECURSIVE CHUNKING:
- Preserves semantic structure better.
- Most commonly used in production RAG systems.
- Provides better retrieval quality.
""")

In [10]:
!pip install sentence-transformers

In [11]:
from sentence_transformers import SentenceTransformer

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings
embeddings = model.encode(chunks)

# Display results
for i, (chunk, embedding) in enumerate(zip(chunks, embeddings), start=1):

    print("="*70)
    print(f"Chunk {i}")
    print("="*70)

    print("Chunk Content:")
    print(chunk)

    print("\nEmbedding Shape:")
    print(embedding.shape)

    print("\nSample Embedding Values (First 20):")
    print(embedding[:20])

    print("\n")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Chunk 1
Chunk Content:
Employee Handbook
Employees are expected to maintain professional conduct.
Office timings are 9 AM to 6 PM.
Employees must follow company ethics and compliance policies.
Leave Policy
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
Travel Policy
Travel expenses are reimbursed within 30 days.
Original receipts must be submitted for reimbursement.
Business travel must be approved by the reporting manager.

Embedding Shape:
(384,)

Sample Embedding Values (First 20):
[ 0.07619021  0.05263857  0.04149914  0.03763472  0.11552272  0.07631592
  0.02893716 -0.03289672 -0.01440236  0.04597221  0.07713117  0.0130358
 -0.05866838  0.02311129 -0.02267209  0.01742607 -0.01888494 -0.04107733
 -0.02090096 -0.02570443]


Chunk 2
Chunk Content:
Business travel must be approved by the reporting manager.
Work From Home Policy
Employees may work from home twice per week.
Manager approval is requir

In [12]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 49.3 MB/s eta 0:00:00


In [13]:
#faiss vectordb
import faiss
import numpy as np

# Convert embeddings to float32
embedding_matrix = np.array(embeddings).astype("float32")

# Get embedding dimension
dimension = embedding_matrix.shape[1]

# Create FAISS index
index = faiss.IndexFlatL2(dimension)

# Add embeddings to index
index.add(embedding_matrix)

# Display information
print("="*50)
print("FAISS VECTOR DATABASE")
print("="*50)

print("Number of Chunks:")
print(len(chunks))

print("\nNumber of Stored Vectors:")
print(index.ntotal)

print("\nEmbedding Dimension:")
print(dimension)

FAISS VECTOR DATABASE
Number of Chunks:
2

Number of Stored Vectors:
2

Embedding Dimension:
384


In [14]:
print("FAISS Index Size:", index.ntotal)

FAISS Index Size: 2


In [17]:
#semantic retrieval
queries = [
    "How many casual leaves are available?",
    "Can employees work from home?",
    "How does travel reimbursement work?",
    "Who is covered under medical insurance?"
]

for query in queries:

    print("\n" + "="*80)
    print("QUERY:", query)
    print("="*80)

    query_embedding = model.encode([query]).astype("float32")

    distances, indices = index.search(
        query_embedding,
        k=3
    )

    for rank, idx in enumerate(indices[0], start=1):

        similarity = 1 / (1 + distances[0][rank-1])

        print(f"\nTop {rank} Chunk:")
        print(chunks[idx])

        print("Similarity Score:",
              round(float(similarity),4))


QUERY: How many casual leaves are available?

Top 1 Chunk:
Employee Handbook
Employees are expected to maintain professional conduct.
Office timings are 9 AM to 6 PM.
Employees must follow company ethics and compliance policies.
Leave Policy
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
Travel Policy
Travel expenses are reimbursed within 30 days.
Original receipts must be submitted for reimbursement.
Business travel must be approved by the reporting manager.
Similarity Score: 0.4311

Top 2 Chunk:
Business travel must be approved by the reporting manager.
Work From Home Policy
Employees may work from home twice per week.
Manager approval is required for additional remote work.
Employees must remain available during working hours.
Medical Insurance Policy
All employees are covered under company medical insurance.
Dependent coverage is available for spouse and children.
Emergency hospitalization exp

In [18]:
!pip install sentence-transformers
!pip install transformers accelerate torch
!pip install faiss-cpu

####RAG Pipeline with Phi-2

In [19]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# =====================================================
# STEP 1: POLICY CHUNKS
# =====================================================

chunks = [

    "Employees are expected to maintain professional conduct. Office timings are 9 AM to 6 PM.",

    "Employees receive 12 casual leaves annually. Employees receive 15 sick leaves annually. Unused casual leaves cannot be carried forward.",

    "Travel expenses are reimbursed within 30 days. Original receipts must be submitted for reimbursement. Business travel requires manager approval.",

    "Employees may work from home twice per week. Manager approval is required for additional remote work.",

    "All employees are covered under company medical insurance. Dependent coverage is available for spouse and children."
]

# =====================================================
# STEP 2: LOAD EMBEDDING MODEL
# =====================================================

print("Loading Embedding Model...")

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

# =====================================================
# STEP 3: CREATE EMBEDDINGS
# =====================================================

print("Generating Embeddings...")

chunk_embeddings = embedding_model.encode(
    chunks
).astype("float32")

print("Embedding Shape:",
      chunk_embeddings.shape)

# =====================================================
# STEP 4: BUILD FAISS INDEX
# =====================================================

dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

index.add(chunk_embeddings)

print("FAISS Index Created")

print("Stored Vectors:",
      index.ntotal)

# =====================================================
# STEP 5: LOAD PHI-2 MODEL
# =====================================================

print("Loading Phi-2 LLM...")

model_name = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Phi-2 Loaded Successfully")

# =====================================================
# STEP 6: EMPLOYEE QUESTIONS
# =====================================================

queries = [

    "How many casual leaves are available?",

    "Can employees work from home?",

    "How does travel reimbursement work?",

    "Who is covered under medical insurance?"
]

# =====================================================
# STEP 7: RETRIEVAL + GENERATION
# =====================================================

for query in queries:

    print("\n")
    print("="*80)
    print("QUESTION:")
    print(query)
    print("="*80)

    # ----------------------------------
    # Query Embedding
    # ----------------------------------

    query_embedding = embedding_model.encode(
        [query]
    ).astype("float32")

    # ----------------------------------
    # Retrieve Top 3 Chunks
    # ----------------------------------

    distances, indices = index.search(
        query_embedding,
        k=3
    )

    retrieved_chunks = []

    print("\nTOP 3 RETRIEVED CHUNKS:\n")

    for rank, idx in enumerate(indices[0], start=1):

        chunk = chunks[idx]

        retrieved_chunks.append(chunk)

        similarity = 1 / (1 + distances[0][rank-1])

        print(f"Rank {rank}")
        print(chunk)

        print(
            "Similarity Score:",
            round(float(similarity),4)
        )

        print("-"*50)

    # ----------------------------------
    # Create Context
    # ----------------------------------

    context = "\n".join(
        retrieved_chunks
    )

    print("\nRETRIEVED CONTEXT:\n")
    print(context)

    # ----------------------------------
    # Prompt
    # ----------------------------------

    prompt = f"""
Context:
{context}

Question:
{query}

Answer:
"""

    # ----------------------------------
    # Generate Answer
    # ----------------------------------

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(llm.device)

    outputs = llm.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.3,
        do_sample=True
    )

    generated_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    answer = generated_text.split(
        "Answer:"
    )[-1].strip()

    print("\nGENERATED ANSWER:\n")
    print(answer)

    print("\n")

Loading Embedding Model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generating Embeddings...
Embedding Shape: (5, 384)
FAISS Index Created
Stored Vectors: 5
Loading Phi-2 LLM...


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.34k [00:00<?, ?B/s]

...


GENERATED ANSWER:

The employee has used 10 sick leaves, so they have 5 sick leaves remaining.

Exercise




QUESTION:
Can employees work from home?

TOP 3 RETRIEVED CHUNKS:

Rank 1
Employees may work from home twice per week. Manager approval is required for additional remote work.
Similarity Score: 0.5998
--------------------------------------------------
Rank 2
All employees are covered under company medical insurance. Dependent coverage is available for spouse and children.
Similarity Score: 0.4529
--------------------------------------------------
Rank 3
Employees are expected to maintain professional conduct. Office timings are 9 AM to 6 PM.
Similarity Score: 0.4119
--------------------------------------------------

RETRIEVED CONTEXT:

Employees may work from home twice per week. Manager approval is required for additional remote work.
All employees are covered under company medical insurance. Dependent coverage is available for spouse and children.
Employees are expected to ma

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



GENERATED ANSWER:

No, employees are not allowed to




QUESTION:
How does travel reimbursement work?

TOP 3 RETRIEVED CHUNKS:

Rank 1
Travel expenses are reimbursed within 30 days. Original receipts must be submitted for reimbursement. Business travel requires manager approval.
Similarity Score: 0.6132
--------------------------------------------------
Rank 2
All employees are covered under company medical insurance. Dependent coverage is available for spouse and children.
Similarity Score: 0.381
--------------------------------------------------
Rank 3
Employees receive 12 casual leaves annually. Employees receive 15 sick leaves annually. Unused casual leaves cannot be carried forward.
Similarity Score: 0.3802
--------------------------------------------------

RETRIEVED CONTEXT:

Travel expenses are reimbursed within 30 days. Original receipts must be submitted for reimbursement. Business travel requires manager approval.
All employees are covered under company medical insurance. D

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



GENERATED ANSWER:

All employees are covered under company medical insurance. However, dependent coverage is only available for spouse and children.

Question:
How many casual leaves are employees




QUESTION:
Who is covered under medical insurance?

TOP 3 RETRIEVED CHUNKS:

Rank 1
All employees are covered under company medical insurance. Dependent coverage is available for spouse and children.
Similarity Score: 0.57
--------------------------------------------------
Rank 2
Employees receive 12 casual leaves annually. Employees receive 15 sick leaves annually. Unused casual leaves cannot be carried forward.
Similarity Score: 0.3576
--------------------------------------------------
Rank 3
Travel expenses are reimbursed within 30 days. Original receipts must be submitted for reimbursement. Business travel requires manager approval.
Similarity Score: 0.3456
--------------------------------------------------

RETRIEVED CONTEXT:

All employees are covered under company medical insurance

In [20]:
# ==========================================
# COMPLETE RAG PIPELINE
# ==========================================

def employee_policy_assistant(question):

    print("\n" + "="*80)
    print("EMPLOYEE QUESTION")
    print("="*80)
    print(question)

    # --------------------------------------
    # STEP 1: EMBEDDING
    # --------------------------------------

    query_embedding = embedding_model.encode(
        [question]
    ).astype("float32")

    print("\n")
    print("QUESTION")
    print("↓")
    print("EMBEDDING")

    # --------------------------------------
    # STEP 2: FAISS SEARCH
    # --------------------------------------

    distances, indices = index.search(
        query_embedding,
        k=3
    )

    print("↓")
    print("FAISS SEARCH")

    # --------------------------------------
    # STEP 3: TOP K CHUNKS
    # --------------------------------------

    retrieved_chunks = []

    print("↓")
    print("TOP-K CHUNKS")

    for i, idx in enumerate(indices[0], start=1):

        retrieved_chunks.append(
            chunks[idx]
        )

        similarity = 1/(1+distances[0][i-1])

        print(f"\nChunk {i}:")
        print(chunks[idx])

        print(
            "Similarity Score:",
            round(float(similarity),4)
        )

    # --------------------------------------
    # STEP 4: CONTEXT CREATION
    # --------------------------------------

    context = "\n".join(
        retrieved_chunks
    )

    # --------------------------------------
    # STEP 5: LLM
    # --------------------------------------

    print("\n↓")
    print("LLM")

    prompt = f"""
Context:
{context}

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(llm.device)

    outputs = llm.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.3,
        do_sample=True
    )

    generated_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    answer = generated_text.split(
        "Answer:"
    )[-1].strip()

    # --------------------------------------
    # STEP 6: FINAL ANSWER
    # --------------------------------------

    print("↓")
    print("ANSWER")

    print("\n")
    print("="*80)
    print("FINAL HUMAN-READABLE ANSWER")
    print("="*80)

    print(answer)

    return answer

In [21]:
employee_policy_assistant(
    "Can employees work from home?"
)

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



EMPLOYEE QUESTION
Can employees work from home?


QUESTION
↓
EMBEDDING
↓
FAISS SEARCH
↓
TOP-K CHUNKS

Chunk 1:
Employees may work from home twice per week. Manager approval is required for additional remote work.
Similarity Score: 0.5998

Chunk 2:
All employees are covered under company medical insurance. Dependent coverage is available for spouse and children.
Similarity Score: 0.4529

Chunk 3:
Employees are expected to maintain professional conduct. Office timings are 9 AM to 6 PM.
Similarity Score: 0.4119

↓
LLM
↓
ANSWER


FINAL HUMAN-READABLE ANSWER
The logical fallacy in this statement is a false dichotomy. It presents only two extreme options and assumes that not supporting the policy automatically means hating the country.

Exercise 5:


'The logical fallacy in this statement is a false dichotomy. It presents only two extreme options and assumes that not supporting the policy automatically means hating the country.\n\nExercise 5:'